# Shreve Week 06 — Black-Scholes-Merton Model

**Week 6** — Black-Scholes-Merton Model

This notebook teaches **black-scholes-merton model** in the style of our graduate probability notebook: precise definitions from Shreve, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve |
|------|-------|--------|
| 1 | **BSM PDE** | Ch. 4.5 |
| 2 | **European call formula** | Ch. 4.5 |
| 3 | **Delta hedging** | Ch. 4.5 |
| 4 | **Put-call parity** | Ch. 4.5 |
| — | **Problem set** | Ch. 4 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. Primary reference: **Shreve**, *Stochastic Calculus for Finance II* — see chapter pointers in each section.

**Shreve reference:** Ch. 4 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — The BSM Partial Differential Equation

Under risk-neutral measure: $dS_t = r S_t\, dt + \sigma S_t\, dW_t$.

For derivative $V(t,S)$, Itô + self-financing portfolio gives:

$$\frac{\partial V}{\partial t} + rS\frac{\partial V}{\partial S} + \frac{1}{2}\sigma^2 S^2 \frac{\partial^2 V}{\partial S^2} = rV$$

**Shreve Ch. 4.5:** derivation of BSM PDE via hedging.


In [ ]:
def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*stats.norm.cdf(d1) - K*np.exp(-r*T)*stats.norm.cdf(d2)

S, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.2
C = black_scholes_call(S, K, T, r, sigma)
print(f"BS call price C = {C:.4f}")


---
# Part 2 — European Call Closed Form

$$C = S_0 \Phi(d_1) - K e^{-rT} \Phi(d_2)$$

where $d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$, $d_2 = d_1 - \sigma\sqrt{T}$.

**Shreve Ch. 4.5:** solving PDE / risk-neutral expectation.


In [ ]:
# Monte Carlo under risk-neutral measure
rng = np.random.default_rng(50)
S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.2
n = 500_000
Z = rng.standard_normal(n)
S_T = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
payoff = np.maximum(S_T - K, 0)
mc_price = np.exp(-r*T) * payoff.mean()
bs_price = black_scholes_call(S0, K, T, r, sigma)
print(f"MC price     = {mc_price:.4f}")
print(f"BS formula   = {bs_price:.4f}")


---
# Part 3 — Delta Hedging

$\Delta = \frac{\partial C}{\partial S} = \Phi(d_1)$. Hold $\Delta$ shares + bond to replicate call.

**Shreve Ch. 4.5:** delta-hedging argument for PDE.


In [ ]:
def bs_delta(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    return stats.norm.cdf(d1)

# Simple delta hedge simulation
rng = np.random.default_rng(51)
n_steps = 252
dt = T / n_steps
S_path = [S0]
for _ in range(n_steps):
    dW = rng.normal(0, np.sqrt(dt))
    S_path.append(S_path[-1] * np.exp((r-0.5*sigma**2)*dt + sigma*dW))
S_path = np.array(S_path)
deltas = [bs_delta(s, K, T - i*dt, r, sigma) for i, s in enumerate(S_path)]
print(f"Delta at S0: {deltas[0]:.4f}")
print(f"Delta at end: {deltas[-1]:.4f} (should → 0 or 1)")


---
# Part 4 — Put-Call Parity

$$C - P = S_0 - K e^{-rT}$$

No arbitrage relationship between European call and put.

**Shreve Ch. 4.5:** put-call parity.


In [ ]:
def black_scholes_put(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return K*np.exp(-r*T)*stats.norm.cdf(-d2) - S*stats.norm.cdf(-d1)

C = black_scholes_call(S0, K, T, r, sigma)
P = black_scholes_put(S0, K, T, r, sigma)
parity = C - P - (S0 - K*np.exp(-r*T))
print(f"C - P           = {C-P:.4f}")
print(f"S - Ke^{-rT}    = {S0 - K*np.exp(-r*T):.4f}")
print(f"Parity residual = {parity:.6f}")


**Your turn:** Plot call price vs $S_0$ and overlay intrinsic value $\max(S-K,0)$.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Derive the BSM PDE for $V(t,S)$ using Itô and a delta-hedged portfolio.

2. Verify put-call parity from the BS formulas.

3. What is $\Delta$ for a deep ITM call as $T \to 0$?

4. Compute $\Gamma = \partial^2 C / \partial S^2$ and interpret.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** Portfolio $\Pi = V - \Delta S$; apply Itô, choose $\Delta = \partial V/\partial S$ to kill $dW$, bond position $r\Pi$.

**2.** Algebra using $\Phi(-x) = 1 - \Phi(x)$.

**3.** $\Delta \to 1$ for $S > K$.

**4.** $\Gamma = \Phi'(d_1)/(S\sigma\sqrt{T})$; sensitivity of delta to spot.

</details>


---
## Further reading

- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 4 — primary text for this week.
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
